In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.5 Paged attention

A contiguous KV cache wastes memory: you reserve max-length for every sequence.
**Paged attention** (vLLM) stores the cache in fixed-size pages, like OS virtual
memory, allocating pages on demand and even sharing them across sequences with a
common prefix.

In [ ]:
page_size = 4
seq_len = 10
n_pages = (seq_len + page_size - 1) // page_size      # ceil division
used = seq_len
reserved = n_pages * page_size
print(f"seq {seq_len} -> {n_pages} pages of {page_size}")
print(f"slots used {used}, reserved {reserved}, wasted {reserved - used}")

Waste is bounded by one page per sequence instead of the full max length, and pages
from a shared prompt prefix can be reused across requests. That is how vLLM serves
many concurrent users on one GPU.

In [ ]:
assert n_pages == 3
assert reserved - used < page_size      # at most one partial page wasted